# PI3Kα (4JPS) — Step 8: Dock the ten shortlisted analogs

**Project:** Tryptanthrin anticancer drug discovery | **Target:** PI3Kα / PIK3CA  
**Input files:** `PI3Kalpha_4JPS_Step6_Redocking.zip` and `PI3Kalpha_Step7_Ligand_Preparation.zip`. Upload both together.  
**Purpose:** Dock the exact ten shortlisted analogs with the receptor and grid used for successful alpelisib redocking. Preserve compound identities, full docked PDBQT files, logs and scores.

**Preconditions:** Step 6 reference top-pose RMSD = 0.6039 Å; Step 7 = 10/10 ligand preparations. The receptor preparation excluded ten distant problematic residues; ligand protonation/tautomer states remain an assumption. A docking score is **not** a measured binding affinity or confirmed inhibitor activity. This notebook does not perform interaction analysis or final ranking.

## 1. Install packages and obtain the official Vina 1.2.7 Linux executable

In [1]:
%pip -q install rdkit pandas numpy requests
from pathlib import Path
from google.colab import files
import os, sys, subprocess, zipfile, json, shutil, hashlib, requests
import pandas as pd
from rdkit import Chem

vina_exec = Path('/content/vina_1.2.7_linux_x86_64')
vina_url = 'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64'
if not vina_exec.exists():
    r = requests.get(vina_url, timeout=120)
    r.raise_for_status()
    vina_exec.write_bytes(r.content)
vina_exec.chmod(0o755)
version_check = subprocess.run([str(vina_exec), '--version'], capture_output=True, text=True)
version_text = version_check.stdout + version_check.stderr
print('Vina:', version_text.strip())
if version_check.returncode or '1.2.7' not in version_text:
    raise RuntimeError('Expected AutoDock Vina 1.2.7; stop and review installation')

Vina: AutoDock Vina v1.2.7


## 2. Upload the redocking and ligand-preparation ZIPs

The cell reads only the required files, checks both archives for corruption, and verifies that the ten ligand PDBQT files match the selected Analog_IDs. It accepts ZIP filename suffixes such as `(1)` from Colab.

In [2]:
out = Path('PI3Kalpha_4JPS_Step8')
(out/'ligands').mkdir(parents=True, exist_ok=True)
(out/'results').mkdir(parents=True, exist_ok=True)
(out/'logs').mkdir(parents=True, exist_ok=True)
print('Upload TWO ZIPs: Step6_Redocking and Step7_Ligand_Preparation')
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 2:
    raise RuntimeError(f'Expected exactly two ZIPs, got {archives}')

step6 = None
step7 = None
for name in archives:
    with zipfile.ZipFile(name) as z:
        bad = z.testzip()
        if bad is not None:
            raise RuntimeError(f'Corrupted file in {name}: {bad}')
        contents = {Path(p).name for p in z.namelist()}
        if {'pi3ka_step6_redocking_report.json', '4JPS_AB_distance_filtered.pdbqt'} <= contents:
            if step6 is not None: raise RuntimeError('Two redocking archives detected')
            step6 = name
        elif {'PI3Kalpha_selected_top10.csv', 'PI3Kalpha_step7_report.json'} <= contents:
            if step7 is not None: raise RuntimeError('Two ligand archives detected')
            step7 = name
        else:
            raise RuntimeError(f'Unrecognized archive contents in {name}')
if step6 is None or step7 is None:
    raise RuntimeError('Step 6 and Step 7 archives are both required')

def unique_member(z, basename):
    hits = [m for m in z.namelist() if not m.endswith('/') and Path(m).name == basename]
    if len(hits) != 1: raise RuntimeError(f'{basename} occurs {len(hits)} times; expected exactly 1')
    return z.read(hits[0])

with zipfile.ZipFile(step6) as z:
    report = json.loads(unique_member(z, 'pi3ka_step6_redocking_report.json'))
    receptor_bytes = unique_member(z, '4JPS_AB_distance_filtered.pdbqt')
    (out/'4JPS_AB_distance_filtered.pdbqt').write_bytes(receptor_bytes)
    (out/'pi3ka_step6_redocking_report.json').write_text(json.dumps(report, indent=2))

with zipfile.ZipFile(step7) as z:
    prep = json.loads(unique_member(z, 'PI3Kalpha_step7_report.json'))
    shortlist_bytes = unique_member(z, 'PI3Kalpha_selected_top10.csv')
    (out/'PI3Kalpha_selected_top10.csv').write_bytes(shortlist_bytes)
    shortlist = pd.read_csv(out/'PI3Kalpha_selected_top10.csv')
    status = pd.read_csv(__import__('io').BytesIO(unique_member(z,'PI3Kalpha_ligand_preparation_status.csv')))
    for analog_id in shortlist['Analog_ID'].astype(str):
        filename = f'{analog_id}.pdbqt'
        data = unique_member(z, filename)
        if b'ROOT' not in data or b'TORSDOF' not in data:
            raise RuntimeError(f'Ligand file is not a recognizable Meeko PDBQT: {filename}')
        (out/'ligands'/filename).write_bytes(data)

assert report['pdb_id'] == '4JPS' and report['top_ranked_pass_2A'] is True
assert float(report['top_ranked_pose_rmsd_A']) <= 2.0
assert report['vina_version'] == '1.2.7' and report['scoring_function'] == 'vina'
assert prep['target'] == 'PI3Kalpha' and prep['all_10_ready'] is True
assert len(shortlist) == 10 and shortlist['Analog_ID'].nunique() == 10
assert len(status) == 10 and bool(status['pdbqt_ok'].all())
assert set(shortlist['Analog_ID'].astype(str)) == set(status['Analog_ID'].astype(str))
for column in ['Analog_ID','SMILES']:
    assert column in shortlist, f'Missing shortlist field {column}'
print('PASS: reference RMSD',report['top_ranked_pose_rmsd_A'],'Å; 10 unique ligand PDBQT files verified')
print('Receptor SHA256:',hashlib.sha256(receptor_bytes).hexdigest())
display(shortlist[['Analog_ID','SMILES']])

Upload TWO ZIPs: Step6_Redocking and Step7_Ligand_Preparation


Saving PI3Kalpha_4JPS_Step6_Redocking.zip to PI3Kalpha_4JPS_Step6_Redocking.zip
Saving PI3Kalpha_Step7_Ligand_Preparation.zip to PI3Kalpha_Step7_Ligand_Preparation.zip
PASS: reference RMSD 0.6039 Å; 10 unique ligand PDBQT files verified
Receptor SHA256: 1bf47f23a914d0d2b0fd8a4423ff4a3c1499fa7c5b36ee6b7222cfe3e31301a4


,Analog_ID,SMILES
0,TRY_c2155e898021da1c,O=C1c2cc(O)ccc2-n2c1nc1c(N3CCOCC3)cccc1c2=O
1,TRY_c6b27d68ce6ec718,CC(=O)Nc1ccc2c(c1)-n1c(nc3c(N4CCOCC4)cccc3c1=O...
2,TRY_40b0c0e6de99494a,COc1ccc2c(c1)C(=O)c1nc3c(N4CCOCC4)cccc3c(=O)n1-2
3,TRY_f751442f427b1d4d,O=C1c2c(N3CCOCC3)cccc2-n2c1nc1c(O)cccc1c2=O
4,TRY_6db5e7a7802259ab,O=C1c2cc(C3CC3)ccc2-n2c1nc1cc(N3CCOCC3)ccc1c2=O
5,TRY_6bf367653fed7969,O=C1c2c(O)cccc2-n2c1nc1cccc(N3CCOCC3)c1c2=O
6,TRY_c88335488290afb4,NC(=O)c1ccc2c(c1)C(=O)c1nc3cc(N4CCOCC4)ccc3c(=...
7,TRY_e514ff1b3f600530,COc1ccc2nc3n(c(=O)c2c1)-c1c(cccc1N1CCOCC1)C3=O
8,TRY_bafc3535a3d34b71,COc1cccc2c(=O)n3c(nc12)C(=O)c1cc(N2CCOCC2)ccc1-3
9,TRY_783966fbcbaa4f5c,CC(C)c1c(N2CCOCC2)ccc2c1-n1c(nc3ccccc3c1=O)C2=O


## 3. Perform ten independent Vina docking runs

The grid, scoring function, exhaustiveness, modes and seed are copied from Step 6. All failures are recorded; successful runs keep full multi-pose PDBQT outputs. Use *Run all* or execute this cell once. On reruns, previous per-ligand logs and outputs are overwritten.

In [3]:
CENTER = [float(x) for x in report['center_A']]
SIZE = [float(x) for x in report['size_A']]
EXHAUSTIVENESS = int(report['exhaustiveness'])
MODES = int(report['n_poses_requested'])
SEED = int(report['seed'])
RECEPTOR = out/'4JPS_AB_distance_filtered.pdbqt'

def extract_energies(path):
    energies = []
    for line in path.read_text().splitlines():
        if line.startswith('REMARK VINA RESULT:'):
            vals = line.split(':',1)[1].split()
            energies.append(float(vals[0]))
    return energies

rows=[]
for i, rec in enumerate(shortlist.to_dict('records'), 1):
    analog_id = str(rec['Analog_ID'])
    ligand = out/'ligands'/f'{analog_id}.pdbqt'
    docked = out/'results'/f'{analog_id}_docked.pdbqt'
    log_path = out/'logs'/f'{analog_id}_vina.log'
    cmd = [str(vina_exec), '--receptor', str(RECEPTOR), '--ligand', str(ligand),
           '--center_x', str(CENTER[0]), '--center_y', str(CENTER[1]), '--center_z', str(CENTER[2]),
           '--size_x', str(SIZE[0]), '--size_y', str(SIZE[1]), '--size_z', str(SIZE[2]),
           '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(MODES),
           '--seed', str(SEED), '--cpu', '2', '--scoring', 'vina', '--out', str(docked)]
    print(f'[{i}/10] Docking {analog_id} ...', flush=True)
    try:
        p = subprocess.run(cmd, text=True, capture_output=True, timeout=1800)
        output = (p.stdout or '') + '\n' + (p.stderr or '')
        log_path.write_text(' '.join(cmd)+'\n\n'+output+'\nexit='+str(p.returncode)+'\n')
        energies = extract_energies(docked) if p.returncode == 0 and docked.exists() else []
        success = p.returncode == 0 and len(energies) > 0
        err = '' if success else f'exit={p.returncode}; energies={len(energies)}; log={log_path.name}'
    except Exception as exc:
        energies=[];success=False;err=str(exc)
        log_path.write_text(err)
    rows.append({'Analog_ID':analog_id,'Selection_rank':rec.get('Selection_rank'),
                 'vina_best_affinity_kcal_mol':energies[0] if energies else None,
                 'pose_count':len(energies),'docking_ok':bool(success),
                 'docked_pdbqt':docked.name if success else '', 'error':err})
    print('  PASS' if success else '  FAIL', 'score:', energies[0] if energies else 'N/A', flush=True)

results = pd.DataFrame(rows)
results.to_csv(out/'results'/'PI3Kalpha_ML_shortlist_docking_results.csv',index=False)
display(results)
print('Docking successes:',int(results['docking_ok'].sum()),'of 10')

[1/10] Docking TRY_c2155e898021da1c ...
  PASS score: -9.383
[2/10] Docking TRY_c6b27d68ce6ec718 ...
  PASS score: -10.175
[3/10] Docking TRY_40b0c0e6de99494a ...
  PASS score: -9.28
[4/10] Docking TRY_f751442f427b1d4d ...
  PASS score: -9.17
[5/10] Docking TRY_6db5e7a7802259ab ...
  PASS score: -10.29
[6/10] Docking TRY_6bf367653fed7969 ...
  PASS score: -9.473
[7/10] Docking TRY_c88335488290afb4 ...
  PASS score: -10.261
[8/10] Docking TRY_e514ff1b3f600530 ...
  PASS score: -8.91
[9/10] Docking TRY_bafc3535a3d34b71 ...
  PASS score: -9.288
[10/10] Docking TRY_783966fbcbaa4f5c ...
  PASS score: -8.716


,Analog_ID,Selection_rank,vina_best_affinity_kcal_mol,pose_count,docking_ok,docked_pdbqt,error
0,TRY_c2155e898021da1c,1,-9.383,9,True,TRY_c2155e898021da1c_docked.pdbqt,
1,TRY_c6b27d68ce6ec718,2,-10.175,9,True,TRY_c6b27d68ce6ec718_docked.pdbqt,
2,TRY_40b0c0e6de99494a,3,-9.280,9,True,TRY_40b0c0e6de99494a_docked.pdbqt,
3,TRY_f751442f427b1d4d,4,-9.170,8,True,TRY_f751442f427b1d4d_docked.pdbqt,
4,TRY_6db5e7a7802259ab,5,-10.290,8,True,TRY_6db5e7a7802259ab_docked.pdbqt,
5,TRY_6bf367653fed7969,6,-9.473,9,True,TRY_6bf367653fed7969_docked.pdbqt,
6,TRY_c88335488290afb4,7,-10.261,9,True,TRY_c88335488290afb4_docked.pdbqt,
7,TRY_e514ff1b3f600530,8,-8.910,9,True,TRY_e514ff1b3f600530_docked.pdbqt,
8,TRY_bafc3535a3d34b71,9,-9.288,9,True,TRY_bafc3535a3d34b71_docked.pdbqt,
9,TRY_783966fbcbaa4f5c,10,-8.716,9,True,TRY_783966fbcbaa4f5c_docked.pdbqt,


Docking successes: 10 of 10


## 4. Generate review table, audit report and downloadable ZIP

The review table joins docking results back to the untouched ML shortlist. It is **not** the final candidate ranking: review interactions and ADMET before prioritizing. Output always records any failed runs.

In [4]:
summary = shortlist.merge(results, on='Analog_ID', how='left', validate='one_to_one', suffixes=('','_docking'))
if not summary['docking_ok'].fillna(False).all():
    print('WARNING: Some ligands failed docking. Review logs before candidate ranking.')
summary = summary.sort_values('vina_best_affinity_kcal_mol', ascending=True, na_position='last')
summary.to_csv(out/'results'/'PI3Kalpha_docking_with_ML_context.csv',index=False)
report8 = {
    'target':'PI3Kalpha','pdb_id':'4JPS','reference':'1LT_alpelisib',
    'input_redocking_RMSD_A':report['top_ranked_pose_rmsd_A'],
    'reference_pose_recovery_pass':report['top_ranked_pass_2A'],
    'receptor_sha256':hashlib.sha256(receptor_bytes).hexdigest(),
    'shortlist_sha256':hashlib.sha256(shortlist_bytes).hexdigest(),
    'center_A':CENTER,'size_A':SIZE,'vina_version':'1.2.7',
    'exhaustiveness':EXHAUSTIVENESS,'num_modes':MODES,'seed':SEED,'scoring_function':'vina',
    'ligands_requested':10,'ligands_docked_successfully':int(results['docking_ok'].sum()),
    'receptor_preparation_caveat':report['receptor_preparation_caveat'],
    'limitation':'Docking scores are computational ranking signals, not experimental PI3Kalpha inhibition or cross-target affinities',
    'next':'Validate top pose quality, analyze top-three contacts and integrate ML/ADMET/SA scores'
}
(out/'PI3Kalpha_step8_docking_report.json').write_text(json.dumps(report8,indent=2)+'\n')
archive = shutil.make_archive('PI3Kalpha_Step8_Docking','zip',root_dir=out)
print('Saved:',archive,'bytes:',Path(archive).stat().st_size)
files.download(archive)
print('Save the executed notebook too (File > Download > Download .ipynb).')

Saved: /content/PI3Kalpha_Step8_Docking.zip bytes: 302000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Save the executed notebook too (File > Download > Download .ipynb).
